# Strength Scan Overview

## Configuration

In [ ]:
from pathlib import Path
import os

SCAN_ROOT = Path(
    os.environ.get(
        "WWGPT_STRENGTH_SCAN_ROOT",
        "/tmp/wwpgd_strength_scan",
    )
)

In [ ]:
import sys

for repo_src in ((Path.cwd() / "src").resolve(), (Path.cwd().parent / "src").resolve()):
    if repo_src.exists() and str(repo_src) not in sys.path:
        sys.path.insert(0, str(repo_src))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from wwgpt.strength_scan_analysis import resolve_scan_root, analyze_strength_scan
scan_root = resolve_scan_root(SCAN_ROOT)
analysis_dir = analyze_strength_scan(scan_root)
fig_dir = scan_root / "analysis" / "notebook_overview"
fig_dir.mkdir(parents=True, exist_ok=True)
print(scan_root)

## Scan manifest

## Run inventory

## Completion and stability audit

## Pairing audit

## Terminal validation-loss comparison

## Validation-loss trajectories

## Projection magnitude

## Throughput and runtime

## Strength-response summary

## Preliminary interpretation

In [ ]:
import json

manifest = json.loads((scan_root / "scan_manifest.json").read_text())
runs = pd.read_csv(analysis_dir / "strength_scan_runs.csv")
terminal = pd.read_csv(analysis_dir / "strength_scan_terminal.csv")
proj = pd.read_csv(analysis_dir / "strength_scan_projection.csv")
summary = pd.read_csv(analysis_dir / "strength_scan_summary.csv")

display(manifest)
display(runs)
display(runs[runs.complete == True])
display(runs[(runs.stable == False) | (runs.complete == False)])
display(terminal)
display(summary)

if summary.seed_count.max() <= 1:
    print("One-seed scan: confidence intervals are unavailable.")

## Plots

The notebook now renders every overview plot inline and saves the same images under `analysis/notebook_overview/`.

Accuracy plots aggregate every available scan run and visualize seed-to-seed variation with standard-deviation error bars and translucent Bollinger-style bands.


In [ ]:
saved_plots = []

def save_and_show(fig, filename):
    path = fig_dir / filename
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(path)
    display(Image(filename=str(path)))

def empty_plot(message, filename):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True)
    ax.axis("off")
    save_and_show(fig, filename)

def plot_strength_line(df, x, y, filename, title, ylabel=None, xlabel="strength"):
    if df.empty or y not in df or x not in df:
        empty_plot(f"No data available for {title}.", filename)
        return
    plot_df = df[[x, y]].dropna().sort_values(x)
    if plot_df.empty:
        empty_plot(f"No finite data available for {title}.", filename)
        return
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(plot_df[x], plot_df[y], marker="o")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--") if "diff" in filename else None
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or y)
    ax.grid(True, alpha=0.25)
    save_and_show(fig, filename)

In [ ]:
plot_strength_line(
    terminal,
    "strength",
    "wwpgd_minus_adamw_final_loss",
    "final_loss_diff.png",
    "WW-PGD minus AdamW final validation loss by strength",
    "final validation-loss difference",
)
plot_strength_line(
    terminal,
    "strength",
    "wwpgd_minus_adamw_minimum_loss",
    "min_loss_diff.png",
    "WW-PGD minus AdamW minimum validation loss by strength",
    "minimum validation-loss difference",
)

## Per-step metrics across strengths

Training/test accuracy, loss, perplexity, bits-per-token, error-rate, generalization-gap, and supporting optimization/runtime metrics are aggregated at each recorded step for every scan. Lines show seed means for each WW-PGD strength (and the AdamW control); shaded Bollinger-style bands show mean ± 2 seed standard deviations, with point error bars shown at the final step for each line.


In [ ]:
trajectory_metrics = [
    ("train_top1_accuracy", "Training top-1 accuracy", "accuracy", (0, 1.05)),
    ("val_top1_accuracy", "Test/validation top-1 accuracy", "accuracy", (0, 1.05)),
    ("train_top5_accuracy", "Training top-5 accuracy", "accuracy", (0, 1.05)),
    ("val_top5_accuracy", "Test/validation top-5 accuracy", "accuracy", (0, 1.05)),
    ("train_loss", "Training loss", "loss", None),
    ("val_loss", "Test/validation loss", "loss", None),
    ("train_minibatch_loss", "Training minibatch loss", "loss", None),
    ("train_perplexity", "Training perplexity", "perplexity", None),
    ("val_perplexity", "Test/validation perplexity", "perplexity", None),
    ("train_bits_per_token", "Training bits per token", "bits/token", None),
    ("val_bits_per_token", "Test/validation bits per token", "bits/token", None),
    ("train_token_error", "Training token error", "error rate", (0, 1.05)),
    ("val_token_error", "Test/validation token error", "error rate", (0, 1.05)),
    ("generalization_gap", "Generalization gap", "val loss - train loss", None),
    ("gradient_norm", "Gradient norm", "gradient norm", None),
    ("learning_rate", "Learning rate", "learning rate", None),
    ("tokens_per_second", "Tokens per second", "tokens/second", None),
    ("examples_per_second", "Examples per second", "examples/second", None),
]


def _scan_label(row):
    if row.optimizer_family == "wwpgd" and pd.notna(row.strength):
        return f"WW-PGD strength={row.strength:g}"
    if row.optimizer_family == "adamw":
        return "AdamW control"
    return str(row.optimizer_family)


def collect_metric_trajectories(runs_df, metric_specs):
    wanted = {name for name, *_ in metric_specs}
    rows = []
    for _, r in runs_df.iterrows():
        mpath = Path(r.run_dir) / "metrics.csv"
        if not mpath.exists():
            continue
        metrics = pd.read_csv(mpath)
        x_col = "step" if "step" in metrics.columns else "tokens_processed" if "tokens_processed" in metrics.columns else None
        if x_col is None:
            continue
        id_cols = [x_col]
        if "tokens_processed" in metrics.columns and x_col != "tokens_processed":
            id_cols.append("tokens_processed")
        present = [m for m in wanted if m in metrics.columns]
        if not present:
            continue
        melted = metrics[id_cols + present].melt(id_vars=id_cols, var_name="metric", value_name="value").dropna(subset=["value"])
        melted["seed"] = r.seed
        melted["strength"] = r.strength
        melted["optimizer_family"] = r.optimizer_family
        melted["scan"] = _scan_label(r)
        melted["x"] = melted[x_col]
        rows.append(melted)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


metric_trajectories = collect_metric_trajectories(runs, trajectory_metrics)


def plot_metric_trajectory(metric, title, ylabel, ylim=None):
    filename = f"{metric}_trajectory_bands.png"
    if metric_trajectories.empty or metric not in set(metric_trajectories.metric):
        empty_plot(f"No per-step data available for {title}.", filename)
        return

    metric_df = metric_trajectories[metric_trajectories.metric == metric].copy()
    metric_df["strength_sort"] = metric_df["strength"].fillna(-1)
    scan_order = (
        metric_df[["scan", "optimizer_family", "strength_sort"]]
        .drop_duplicates()
        .sort_values(["optimizer_family", "strength_sort", "scan"])
    )
    colors = plt.cm.tab10(range(10))
    fig, ax = plt.subplots(figsize=(11, 6))
    plotted = False
    for idx, row in enumerate(scan_order.itertuples(index=False)):
        scan_df = metric_df[metric_df.scan == row.scan]
        agg = (
            scan_df.groupby("x")
            .agg(mean=("value", "mean"), std=("value", "std"), seed_count=("seed", "nunique"))
            .reset_index()
            .sort_values("x")
        )
        if agg.empty:
            continue
        agg["std"] = agg["std"].fillna(0.0)
        band = 2.0 * agg["std"]
        color = colors[idx % len(colors)]
        ax.plot(agg["x"], agg["mean"], marker="o", linewidth=1.8, label=f"{row.scan} mean", color=color)
        ax.fill_between(agg["x"], agg["mean"] - band, agg["mean"] + band, color=color, alpha=0.14)
        last = agg.iloc[-1]
        ax.errorbar([last["x"]], [last["mean"]], yerr=[2.0 * last["std"]], fmt="none", capsize=4, color=color)
        plotted = True

    if not plotted:
        plt.close(fig)
        empty_plot(f"No finite per-step data available for {title}.", filename)
        return
    ax.set_title(f"{title} by training step (mean ± 2 seed std Bollinger bands)")
    ax.set_xlabel("step")
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    save_and_show(fig, filename)


for metric, title, ylabel, ylim in trajectory_metrics:
    plot_metric_trajectory(metric, title, ylabel, ylim)


In [ ]:
if len(proj):
    projection_by_strength = proj.groupby("strength", as_index=False).agg(
        median_relative_frobenius_change=("median_relative_frobenius_change", "mean"),
        total_projection_runtime=("total_projection_runtime", "mean"),
    )
else:
    projection_by_strength = pd.DataFrame()

plot_strength_line(
    projection_by_strength,
    "strength",
    "median_relative_frobenius_change",
    "projection_norm.png",
    "Mean projection magnitude by strength",
    "relative Frobenius change",
)
plot_strength_line(
    projection_by_strength,
    "strength",
    "total_projection_runtime",
    "projection_runtime.png",
    "Mean projection runtime by strength",
    "projection runtime (seconds)",
)

In [ ]:
wwpgd_runs = runs[runs.optimizer_family == "wwpgd"].copy()
if not wwpgd_runs.empty:
    runtime_by_strength = wwpgd_runs.groupby("strength", as_index=False).agg(
        total_immediate_weightwatcher_runtime=("total_immediate_weightwatcher_runtime", "mean"),
        tokens_per_second=("tokens_per_second", "mean"),
        stable=("stable", "mean"),
    )
else:
    runtime_by_strength = pd.DataFrame()

plot_strength_line(
    runtime_by_strength,
    "strength",
    "total_immediate_weightwatcher_runtime",
    "immediate_ww.png",
    "Mean immediate WeightWatcher runtime by strength",
    "runtime (seconds)",
)
plot_strength_line(
    runtime_by_strength,
    "strength",
    "tokens_per_second",
    "tokens_per_second.png",
    "Mean training throughput by strength",
    "tokens per second",
)

if runtime_by_strength.empty:
    empty_plot("No WW-PGD runs are available for stability plotting.", "stability.png")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(runtime_by_strength["strength"].astype(str), runtime_by_strength["stable"])
    ax.set_title("Stable-run fraction by strength")
    ax.set_xlabel("strength")
    ax.set_ylabel("stable fraction")
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis="y", alpha=0.25)
    save_and_show(fig, "stability.png")

In [ ]:
if metric_trajectories.empty:
    empty_plot("No metrics.csv files with per-step trajectories were found.", "metric_trajectory_inventory.png")
else:
    inventory = (
        metric_trajectories.groupby(["metric", "scan"])
        .agg(seed_count=("seed", "nunique"), recorded_points=("value", "size"), final_step=("x", "max"))
        .reset_index()
        .sort_values(["metric", "scan"])
    )
    display(inventory)

    terminal_candidates = metric_trajectories.sort_values("x").groupby(["metric", "scan", "seed"], as_index=False).tail(1)
    terminal_summary = (
        terminal_candidates.groupby(["metric", "scan"])
        .agg(mean=("value", "mean"), std=("value", "std"), seed_count=("seed", "nunique"))
        .reset_index()
    )
    display(terminal_summary[terminal_summary.metric.isin(["val_top1_accuracy", "val_top5_accuracy", "val_loss", "val_perplexity"])])


In [ ]:
plot_manifest = pd.DataFrame({"plot_file": [str(path.relative_to(scan_root)) for path in saved_plots]})
plot_manifest.to_csv(fig_dir / "plot_manifest.csv", index=False)
display(plot_manifest)
print("Observed trade-offs are shown above; no optimal strength is selected automatically.")